# 07 PD Calibration and Final Selection

Milestone 5 calibrates the selected candidate PD model, compares calibrated versus uncalibrated probabilities, and prepares final PD outputs for later Expected Loss work.

## 1. Load Data and Define Candidate Model

In [ ]:
from pathlib import Path
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from sklearn.frozen import FrozenEstimator
except Exception:
    FrozenEstimator = None

In [ ]:
data_path = "../data/processed/loans_clean_v1.csv"
tables_dir = Path("../reports/tables")
figures_dir = Path("../reports/figures")
models_dir = Path("../models")
processed_dir = Path("../data/processed")

tables_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)

loans = pd.read_csv(data_path, low_memory=False)

target_column = "default_flag"

numeric_features = [
    "loan_amnt", "term", "int_rate", "annual_inc", "dti",
    "delinq_2yrs", "revol_util", "total_acc", "open_acc", "emp_length",
]

categorical_features = [
    "grade", "sub_grade", "home_ownership", "verification_status", "purpose", "addr_state",
]

required_columns = [target_column] + numeric_features + categorical_features
missing_columns = [column for column in required_columns if column not in loans.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

loans_model = loans.dropna(subset=[target_column]).copy()
loans_model[target_column] = loans_model[target_column].astype(int)

X = loans_model[numeric_features + categorical_features]
y = loans_model[target_column]

print(f"Rows: {loans_model.shape[0]:,}")
print(f"Columns: {loans_model.shape[1]:,}")
print(f"Default rate: {y.mean():.2%}")

## 2. Create Train / Calibration / Test Split

The train set fits the model, the calibration set fits the calibration layer, and the test set is held untouched for final evaluation.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.40,
    random_state=42,
    stratify=y,
)

X_calibration, X_test, y_calibration, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp,
)

print(f"Training rows: {len(X_train):,}")
print(f"Calibration rows: {len(X_calibration):,}")
print(f"Test rows: {len(X_test):,}")
print(f"Training default rate: {y_train.mean():.2%}")
print(f"Calibration default rate: {y_calibration.mean():.2%}")
print(f"Test default rate: {y_test.mean():.2%}")

## 3. Build Candidate Model Pipeline

In [ ]:
def make_one_hot_encoder():
    """Support older and newer scikit-learn versions."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


def build_candidate_pipeline():
    """Build the selected L2 logistic regression PD pipeline."""
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_one_hot_encoder()),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_transformer, numeric_features),
            ("categorical", categorical_transformer, categorical_features),
        ]
    )

    model = LogisticRegression(
        penalty="l2",
        solver="lbfgs",
        max_iter=1000,
        class_weight=None,
        random_state=42,
    )

    return Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])

In [ ]:
base_model = build_candidate_pipeline()
base_model.fit(X_train, y_train)

print("Candidate L2 logistic regression model fitted on training set only.")

## 4. Create Calibration Methods

This notebook uses the training set for the base model, the calibration set for calibration, and the test set only for final evaluation. If `cv="prefit"` is needed as a fallback, it is used only after trying the newer `FrozenEstimator` approach.

In [ ]:
def fit_calibrated_model(fitted_model, method, X_calib, y_calib):
    """Fit a calibration layer on a fitted model without using the test set."""
    if FrozenEstimator is not None:
        calibrated_model = CalibratedClassifierCV(
            estimator=FrozenEstimator(fitted_model),
            method=method,
        )
        calibrated_model.fit(X_calib, y_calib)
        return calibrated_model

    # Fallback for older scikit-learn versions.
    # cv="prefit" may show a deprecation warning in newer versions, but it is appropriate here
    # because the base model was fitted only on the training set.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", FutureWarning)
        calibrated_model = CalibratedClassifierCV(
            estimator=fitted_model,
            method=method,
            cv="prefit",
        )
        calibrated_model.fit(X_calib, y_calib)
    return calibrated_model


model_versions = {
    "Uncalibrated L2 Logistic Regression": base_model,
    "Sigmoid Calibrated L2 Logistic Regression": fit_calibrated_model(
        fitted_model=base_model,
        method="sigmoid",
        X_calib=X_calibration,
        y_calib=y_calibration,
    ),
    "Isotonic Calibrated L2 Logistic Regression": fit_calibrated_model(
        fitted_model=base_model,
        method="isotonic",
        X_calib=X_calibration,
        y_calib=y_calibration,
    ),
}

list(model_versions.keys())

## 5. Evaluate Calibration and Performance

In [ ]:
def calculate_calibration_slope_intercept(y_true, y_pred_proba):
    """Estimate calibration intercept and slope using a simple logistic recalibration model."""
    clipped_pd = np.clip(y_pred_proba, 1e-6, 1 - 1e-6)
    logit_pd = np.log(clipped_pd / (1 - clipped_pd)).reshape(-1, 1)
    calibration_model = LogisticRegression(solver="lbfgs")
    calibration_model.fit(logit_pd, y_true)

    return calibration_model.intercept_[0], calibration_model.coef_[0][0]


def create_risk_deciles(y_true, y_pred_proba, model_version):
    """Create risk deciles from predicted PDs."""
    results = pd.DataFrame({
        "default_flag": y_true.values,
        "predicted_pd": y_pred_proba,
    })
    results["decile"] = pd.qcut(results["predicted_pd"], q=10, labels=False, duplicates="drop") + 1

    deciles = (
        results.groupby("decile")
        .agg(
            number_of_loans=("default_flag", "size"),
            average_predicted_pd=("predicted_pd", "mean"),
            actual_default_rate=("default_flag", "mean"),
            number_of_defaults=("default_flag", "sum"),
            min_predicted_pd=("predicted_pd", "min"),
            max_predicted_pd=("predicted_pd", "max"),
        )
        .reset_index()
        .sort_values("decile")
    )
    deciles.insert(0, "model_version", model_version)

    return deciles


def create_calibration_bins(y_true, y_pred_proba, model_version):
    """Create 10 calibration bins from predicted PDs."""
    results = pd.DataFrame({
        "default_flag": y_true.values,
        "predicted_pd": y_pred_proba,
    })
    results["bin_number"] = pd.qcut(results["predicted_pd"], q=10, labels=False, duplicates="drop") + 1

    bins = (
        results.groupby("bin_number")
        .agg(
            number_of_loans=("default_flag", "size"),
            average_predicted_pd=("predicted_pd", "mean"),
            actual_default_rate=("default_flag", "mean"),
        )
        .reset_index()
        .sort_values("bin_number")
    )
    bins["difference_actual_minus_predicted"] = bins["actual_default_rate"] - bins["average_predicted_pd"]
    bins.insert(0, "model_version", model_version)

    return bins

In [ ]:
comparison_rows = []
calibration_bin_tables = []
decile_tables = []
test_predictions_by_model = {}

for model_version, model in model_versions.items():
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    test_predictions_by_model[model_version] = y_pred_proba

    calibration_intercept, calibration_slope = calculate_calibration_slope_intercept(y_test, y_pred_proba)
    deciles = create_risk_deciles(y_test, y_pred_proba, model_version)
    calibration_bins = create_calibration_bins(y_test, y_pred_proba, model_version)

    lowest_decile_default_rate = deciles.loc[deciles["decile"].idxmin(), "actual_default_rate"]
    highest_decile_default_rate = deciles.loc[deciles["decile"].idxmax(), "actual_default_rate"]
    decile_ratio = highest_decile_default_rate / lowest_decile_default_rate

    comparison_rows.append({
        "model_version": model_version,
        "roc_auc": roc_auc_score(y_test, y_pred_proba),
        "pr_auc_average_precision": average_precision_score(y_test, y_pred_proba),
        "brier_score": brier_score_loss(y_test, y_pred_proba),
        "log_loss": log_loss(y_test, y_pred_proba),
        "mean_predicted_pd": y_pred_proba.mean(),
        "actual_default_rate": y_test.mean(),
        "mean_pd_minus_actual_default_rate": y_pred_proba.mean() - y_test.mean(),
        "calibration_intercept": calibration_intercept,
        "calibration_slope": calibration_slope,
        "lowest_decile_default_rate": lowest_decile_default_rate,
        "highest_decile_default_rate": highest_decile_default_rate,
        "highest_lowest_decile_ratio": decile_ratio,
    })

    calibration_bin_tables.append(calibration_bins)
    decile_tables.append(deciles)

model_comparison = pd.DataFrame(comparison_rows).sort_values("brier_score")
calibration_bins_all = pd.concat(calibration_bin_tables, ignore_index=True)
deciles_all = pd.concat(decile_tables, ignore_index=True)

model_comparison_path = tables_dir / "pd_calibration_model_comparison_v1.csv"
calibration_bins_path = tables_dir / "pd_calibration_bins_v1.csv"
deciles_path = tables_dir / "pd_calibration_deciles_v1.csv"

model_comparison.to_csv(model_comparison_path, index=False)
calibration_bins_all.to_csv(calibration_bins_path, index=False)
deciles_all.to_csv(deciles_path, index=False)

display(model_comparison)
print(f"Saved model comparison to: {model_comparison_path}")
print(f"Saved calibration bins to: {calibration_bins_path}")
print(f"Saved risk deciles to: {deciles_path}")

## 6. Create Calibration Bins

Calibration bins were created and saved in the previous step as `pd_calibration_bins_v1.csv`.

In [ ]:
calibration_bins_all.head(15)

## 7. Risk Decile Analysis

Risk deciles were created and saved in the evaluation step as `pd_calibration_deciles_v1.csv`.

In [ ]:
deciles_all.head(15)

## 8. Time-Based Calibration Check

In [ ]:
time_validation_rows = []

if "issue_d" in loans_model.columns:
    loans_time = loans_model.copy()
    loans_time["issue_d"] = pd.to_datetime(loans_time["issue_d"], format="%b-%Y", errors="coerce")
    loans_time = loans_time.dropna(subset=["issue_d"]).copy()

    older_loans = loans_time[loans_time["issue_d"] < "2016-10-01"].copy()
    newer_loans = loans_time[loans_time["issue_d"] >= "2016-10-01"].copy()

    X_older = older_loans[numeric_features + categorical_features]
    y_older = older_loans[target_column]
    X_time_test = newer_loans[numeric_features + categorical_features]
    y_time_test = newer_loans[target_column]

    X_time_train, X_time_calibration, y_time_train, y_time_calibration = train_test_split(
        X_older,
        y_older,
        test_size=0.25,
        random_state=42,
        stratify=y_older,
    )

    time_base_model = build_candidate_pipeline()
    time_base_model.fit(X_time_train, y_time_train)

    time_model_versions = {
        "Uncalibrated L2 Logistic Regression": time_base_model,
        "Sigmoid Calibrated L2 Logistic Regression": fit_calibrated_model(
            fitted_model=time_base_model,
            method="sigmoid",
            X_calib=X_time_calibration,
            y_calib=y_time_calibration,
        ),
        "Isotonic Calibrated L2 Logistic Regression": fit_calibrated_model(
            fitted_model=time_base_model,
            method="isotonic",
            X_calib=X_time_calibration,
            y_calib=y_time_calibration,
        ),
    }

    for model_version, model in time_model_versions.items():
        y_time_pred_proba = model.predict_proba(X_time_test)[:, 1]
        time_deciles = create_risk_deciles(y_time_test, y_time_pred_proba, model_version)
        lowest_time_decile_default_rate = time_deciles.loc[time_deciles["decile"].idxmin(), "actual_default_rate"]
        highest_time_decile_default_rate = time_deciles.loc[time_deciles["decile"].idxmax(), "actual_default_rate"]
        time_decile_ratio = highest_time_decile_default_rate / lowest_time_decile_default_rate

        time_validation_rows.append({
            "model_version": model_version,
            "time_train_rows": len(X_time_train),
            "time_calibration_rows": len(X_time_calibration),
            "time_test_rows": len(X_time_test),
            "roc_auc": roc_auc_score(y_time_test, y_time_pred_proba),
            "pr_auc_average_precision": average_precision_score(y_time_test, y_time_pred_proba),
            "brier_score": brier_score_loss(y_time_test, y_time_pred_proba),
            "log_loss": log_loss(y_time_test, y_time_pred_proba),
            "mean_predicted_pd": y_time_pred_proba.mean(),
            "actual_default_rate": y_time_test.mean(),
            "mean_pd_minus_actual_default_rate": y_time_pred_proba.mean() - y_time_test.mean(),
            "highest_lowest_decile_ratio": time_decile_ratio,
        })

    time_validation = pd.DataFrame(time_validation_rows).sort_values("brier_score")
else:
    time_validation = pd.DataFrame()
    print("issue_d does not exist. Time-based calibration requires issue_d in the processed dataset.")

time_validation_path = tables_dir / "pd_calibration_time_validation_v1.csv"
time_validation.to_csv(time_validation_path, index=False)

display(time_validation)
print(f"Saved time validation results to: {time_validation_path}")

## 9. Save Figures

In [ ]:
plt.figure(figsize=(8, 6))
for model_version, y_pred_proba in test_predictions_by_model.items():
    prob_true, prob_pred = calibration_curve(y_test, y_pred_proba, n_bins=10, strategy="quantile")
    plt.plot(prob_pred, prob_true, marker="o", label=model_version)
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect calibration")
plt.xlabel("Average Predicted PD")
plt.ylabel("Actual Default Rate")
plt.title("Calibration Curve Comparison")
plt.legend(fontsize=8)
plt.tight_layout()
calibration_curve_path = figures_dir / "pd_calibration_curve_comparison_v1.png"
plt.savefig(calibration_curve_path, dpi=150)
plt.show()

plt.figure(figsize=(7, 5))
plt.bar(model_comparison["model_version"], model_comparison["brier_score"], color="steelblue")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Brier Score")
plt.title("Brier Score by Calibration Method")
plt.tight_layout()
brier_path = figures_dir / "pd_calibration_brier_score_comparison_v1.png"
plt.savefig(brier_path, dpi=150)
plt.show()

plt.figure(figsize=(7, 5))
plt.bar(model_comparison["model_version"], model_comparison["log_loss"], color="seagreen")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Log Loss")
plt.title("Log Loss by Calibration Method")
plt.tight_layout()
log_loss_path = figures_dir / "pd_calibration_log_loss_comparison_v1.png"
plt.savefig(log_loss_path, dpi=150)
plt.show()

plt.figure(figsize=(8, 6))
for model_version, group in deciles_all.groupby("model_version"):
    plt.plot(group["decile"], group["actual_default_rate"], marker="o", label=model_version)
plt.xlabel("Risk Decile")
plt.ylabel("Actual Default Rate")
plt.title("Default Rate by Risk Decile")
plt.xticks(range(1, 11))
plt.legend(fontsize=8)
plt.tight_layout()
decile_path = figures_dir / "pd_calibration_decile_comparison_v1.png"
plt.savefig(decile_path, dpi=150)
plt.show()

plt.figure(figsize=(8, 6))
for model_version, y_pred_proba in test_predictions_by_model.items():
    plt.hist(y_pred_proba, bins=40, alpha=0.4, label=model_version)
plt.xlabel("Predicted PD")
plt.ylabel("Number of Loans")
plt.title("Predicted PD Distribution by Calibration Method")
plt.legend(fontsize=8)
plt.tight_layout()
distribution_path = figures_dir / "pd_distribution_by_calibration_method_v1.png"
plt.savefig(distribution_path, dpi=150)
plt.show()

## 10. Select Final PD Model Programmatically

In [ ]:
def get_metric_value(table, model_version, metric):
    return table.loc[table["model_version"] == model_version, metric].iloc[0]


baseline_version = "Uncalibrated L2 Logistic Regression"
sigmoid_version = "Sigmoid Calibrated L2 Logistic Regression"
isotonic_version = "Isotonic Calibrated L2 Logistic Regression"

baseline_brier = get_metric_value(model_comparison, baseline_version, "brier_score")
baseline_log_loss = get_metric_value(model_comparison, baseline_version, "log_loss")
baseline_roc_auc = get_metric_value(model_comparison, baseline_version, "roc_auc")
baseline_pr_auc = get_metric_value(model_comparison, baseline_version, "pr_auc_average_precision")

selection_rows = []
for _, row in model_comparison.iterrows():
    selection_rows.append({
        "model_version": row["model_version"],
        "brier_score": row["brier_score"],
        "log_loss": row["log_loss"],
        "roc_auc": row["roc_auc"],
        "pr_auc_average_precision": row["pr_auc_average_precision"],
        "mean_pd_minus_actual_default_rate": row["mean_pd_minus_actual_default_rate"],
        "highest_lowest_decile_ratio": row["highest_lowest_decile_ratio"],
        "brier_improvement_vs_uncalibrated": baseline_brier - row["brier_score"],
        "log_loss_improvement_vs_uncalibrated": baseline_log_loss - row["log_loss"],
        "roc_auc_change_vs_uncalibrated": row["roc_auc"] - baseline_roc_auc,
        "pr_auc_change_vs_uncalibrated": row["pr_auc_average_precision"] - baseline_pr_auc,
    })

selection_table = pd.DataFrame(selection_rows)

selected_model_version = baseline_version
selection_reason = "Uncalibrated L2 logistic regression is retained because calibration did not materially improve probability quality."

sigmoid_brier_improvement = get_metric_value(selection_table, sigmoid_version, "brier_improvement_vs_uncalibrated")
sigmoid_log_loss_improvement = get_metric_value(selection_table, sigmoid_version, "log_loss_improvement_vs_uncalibrated")
sigmoid_roc_change = get_metric_value(selection_table, sigmoid_version, "roc_auc_change_vs_uncalibrated")
sigmoid_pr_change = get_metric_value(selection_table, sigmoid_version, "pr_auc_change_vs_uncalibrated")

isotonic_brier_improvement = get_metric_value(selection_table, isotonic_version, "brier_improvement_vs_uncalibrated")
isotonic_log_loss_improvement = get_metric_value(selection_table, isotonic_version, "log_loss_improvement_vs_uncalibrated")
isotonic_roc_change = get_metric_value(selection_table, isotonic_version, "roc_auc_change_vs_uncalibrated")

if (
    (sigmoid_brier_improvement > 0.0005 or sigmoid_log_loss_improvement > 0.0005)
    and sigmoid_roc_change > -0.001
    and sigmoid_pr_change > -0.001
):
    selected_model_version = sigmoid_version
    selection_reason = "Sigmoid calibration improves probability quality without materially hurting ranking performance."

if (
    isotonic_brier_improvement > 0.0015
    and isotonic_log_loss_improvement > 0.0015
    and isotonic_roc_change > -0.001
):
    selected_model_version = isotonic_version
    selection_reason = "Isotonic calibration materially improves calibration metrics while preserving ranking; review overfitting risk before final use."

selection_table["selected_final_model"] = selection_table["model_version"] == selected_model_version
selection_table["selection_reason"] = np.where(selection_table["selected_final_model"], selection_reason, "Not selected")

selection_path = tables_dir / "pd_final_model_selection_v1.csv"
selection_table.to_csv(selection_path, index=False)

display(selection_table)
print(f"Selected model: {selected_model_version}")
print(f"Reason: {selection_reason}")
print(f"Saved final model selection table to: {selection_path}")

## 11. Save Final Model and Predictions

In [ ]:
final_model = model_versions[selected_model_version]
final_model_path = models_dir / "final_pd_model_v1.pkl"
joblib.dump(final_model, final_model_path)

final_test_predicted_pd = final_model.predict_proba(X_test)[:, 1]
test_predictions = pd.DataFrame({
    "default_flag": y_test.values,
    "predicted_pd": final_test_predicted_pd,
    "model_version": selected_model_version,
})

test_predictions["risk_decile"] = pd.qcut(
    test_predictions["predicted_pd"],
    q=10,
    labels=False,
    duplicates="drop",
) + 1

optional_columns = ["loan_id", "id", "issue_d", "funded_amnt", "recoveries"]
available_optional_columns = [column for column in optional_columns if column in loans_model.columns]
test_metadata = loans_model.loc[X_test.index, available_optional_columns].reset_index(drop=True)
test_predictions = pd.concat([test_metadata, test_predictions.reset_index(drop=True)], axis=1)

test_predictions_path = processed_dir / "final_pd_test_predictions_v1.csv"
test_predictions.to_csv(test_predictions_path, index=False)

loans_with_pd = loans_model.copy()
loans_with_pd["predicted_pd"] = final_model.predict_proba(loans_with_pd[numeric_features + categorical_features])[:, 1]
loans_with_pd["pd_risk_decile"] = pd.qcut(
    loans_with_pd["predicted_pd"],
    q=10,
    labels=False,
    duplicates="drop",
) + 1

loans_with_pd_path = processed_dir / "loans_with_pd_v1.csv"
loans_with_pd.to_csv(loans_with_pd_path, index=False)

print(f"Saved final model to: {final_model_path}")
print(f"Saved test predictions to: {test_predictions_path}")
print(f"Saved full scored dataset to: {loans_with_pd_path}")

Full-dataset predictions are not fully out-of-sample because the final model was trained and calibrated using portions of this dataset. Use `loans_with_pd_v1.csv` carefully for analysis and reporting; use the test-set predictions for cleaner out-of-sample evaluation.

## 12. Final Notebook Checklist

In [ ]:
checklist_files = [
    tables_dir / "pd_calibration_bins_v1.csv",
    tables_dir / "pd_calibration_model_comparison_v1.csv",
    tables_dir / "pd_calibration_deciles_v1.csv",
    tables_dir / "pd_calibration_time_validation_v1.csv",
    tables_dir / "pd_final_model_selection_v1.csv",
    figures_dir / "pd_calibration_curve_comparison_v1.png",
    figures_dir / "pd_calibration_brier_score_comparison_v1.png",
    figures_dir / "pd_calibration_log_loss_comparison_v1.png",
    figures_dir / "pd_calibration_decile_comparison_v1.png",
    figures_dir / "pd_distribution_by_calibration_method_v1.png",
    models_dir / "final_pd_model_v1.pkl",
    processed_dir / "final_pd_test_predictions_v1.csv",
    processed_dir / "loans_with_pd_v1.csv",
]

checklist = pd.DataFrame({
    "file_path": [str(path) for path in checklist_files],
    "created": [path.exists() for path in checklist_files],
})

checklist